# Lab 8: Introduction au Framework ADK et Multi-Provider

**Navigation** : [Index](../../README.md) | [Précédent <<](../../PythonAgentsForDataScience/Day3/Labs/Lab7-Data-Analysis-Agent/Lab7-Data-Analysis-Agent.ipynb) | [Suivant >>](Lab9-First-ADK-Agent.ipynb)

## Objectifs d'apprentissage

A la fin de ce laboratoire, vous saurez :
1. Expliquer l'architecture du Google Agent Development Kit (ADK)
2. Configurer un environnement multi-provider (Gemini, vLLM, OpenAI)
3. Créer un premier client LLM avec votre provider choisi
4. Comparer les réponses de différents providers sur le même prompt

### Prérequis
- Python 3.10+
- Fichier `.env` configuré avec au moins un provider (voir section Configuration)
- Connaissance de base des LLMs (API, tokens, temperature)

### Durée estimée : 45-60 minutes

---

## Configuration requise

Avant de commencer, assurez-vous d'avoir configuré votre fichier `.env` :

```bash
# Exemple de configuration .env
ACTIVE_PROVIDER=gemini  # ou vllm, openai

# Gemini (optionnel)
GOOGLE_API_KEY=your_gemini_key

# vLLM (optionnel)
VLLM_BASE_URL=https://your-vllm-endpoint.com/v1
VLLM_MODEL=Qwen/Qwen2.5-72B-Instruct

# OpenAI (optionnel)
OPENAI_API_KEY=sk-...
```

---

## 1. Architecture Google ADK

Le **Agent Development Kit (ADK)** de Google est un framework pour construire des agents IA avec :

- **Agents** : Entités qui interagissent via des tools
- **Tools** : Fonctions que l'agent peut appeler
- **Sessions** : Gestion de l'état conversationnel
- **Memory** : Persistance du contexte

### Comparaison avec LangChain

| Aspect | ADK | LangChain |
|--------|-----|----------|
| **Philosophie** | Google-first, intégré GCP | Multi-provider natif |
| **Agents** | Agent classes avec tools | Runnable, chains, agents |
| **Mémoire** | Session state intégrée | Modules séparés |
| **Déploiement** | Vertex AI Agent Engine | Variable |
| **Providers** | Gemini natif, OpenAI compatible | 50+ providers |

## 2. Configuration de l'Environnement

### Installation des dépendances

In [1]:
# Installation des dépendances (décommenter si nécessaire)
# !pip install -r ../requirements.txt

Import des modules et verification de l'installation.

In [2]:
import sys
from pathlib import Path

# Add parent directory (AgenticDataScience) to path for config/utils imports
sys.path.insert(0, str(Path().resolve().parent))

from config import get_settings, get_provider_config, ProviderType
from utils import LLMClient, generate
import os

### Vérification de la configuration

In [3]:
# Chargement des settings
settings = get_settings()

print(f"Provider actif: {settings.active_provider}")
print(f"Modèle configuré: {settings.gemini_model if settings.active_provider == 'gemini' else 'autre'}")

Provider actif: openrouter
Modèle configuré: autre


Configuration detaillee du provider LLM.

In [4]:
# Configuration détaillée du provider
config = get_provider_config(settings)
print(f"Provider: {config.provider.value}")
print(f"Modèle: {config.model}")
print(f"Base URL: {config.base_url or 'défaut'}")
print(f"API Key configurée: {'Oui' if config.api_key else 'Non'}")

Provider: openrouter
Modèle: openai/gpt-4.1-mini
Base URL: https://openrouter.ai/api/v1
API Key configurée: Oui


## 3. Premier Test avec le Client LLM

Utilisons notre couche d'abstraction pour envoyer un prompt simple.

In [5]:
# Création du client
client = LLMClient()
print(client)

LLMClient(provider=openrouter, model=openai/gpt-4.1-mini)


Test de generation simple avec le client configure.

In [6]:
# Test de génération simple
response = client.generate(
    "Explique ce qu'est un agent IA en 2 phrases.",
    temperature=0.7
)
print(response)

Un agent IA est un système informatique capable de percevoir son environnement, de prendre des décisions et d'agir de manière autonome pour atteindre des objectifs spécifiques. Il utilise des techniques d'intelligence artificielle, comme l'apprentissage automatique ou le raisonnement, pour s'adapter et améliorer ses performances au fil du temps.


### Test avec prompt système

In [7]:
response = client.generate(
    prompt="Quel est le meilleur algorithme pour classifier des données tabulaires?",
    system="Tu es un expert en machine learning. Réponds de façon concise et technique.",
    temperature=0.3
)
print(response)

Il n’existe pas un "meilleur" algorithme universel pour la classification de données tabulaires, car la performance dépend fortement de la nature des données (taille, bruit, types de variables, déséquilibre des classes, etc.). Cependant, les algorithmes suivants sont généralement performants et largement utilisés :

- **Gradient Boosting Machines (GBM)** : XGBoost, LightGBM, CatBoost — très efficaces, gèrent bien les données hétérogènes et le déséquilibre.
- **Random Forest** : robuste, moins sensible au surapprentissage, facile à interpréter.
- **Logistic Regression** : baseline simple, performant si relations linéaires.
- **Neural Networks (MLP)** : utiles si beaucoup de données et interactions complexes.

En pratique, commencer par un GBM (ex. LightGBM) est souvent un bon choix, suivi d’une validation croisée et d’un tuning hyperparamétrique.


## 4. Comparaison Multi-Provider

Testons le même prompt avec différents providers pour comparer les réponses.

In [8]:
test_prompt = "Donne 3 bonnes pratiques pour nettoyer un dataset."
test_system = "Réponds en français, de façon structurée avec des bullet points."

# Test avec le provider actuel
print(f"=== Provider: {settings.active_provider.upper()} ===")
response = client.generate(test_prompt, system=test_system)
print(response)

=== Provider: OPENROUTER ===


Voici trois bonnes pratiques pour nettoyer un dataset :

- **Gérer les valeurs manquantes :**  
  - Identifier les données absentes ou nulles.  
  - Choisir une stratégie adaptée : suppression des lignes/colonnes concernées, imputation par la moyenne/médiane/mode, ou utilisation d’algorithmes capables de gérer les valeurs manquantes.

- **Traiter les valeurs aberrantes :**  
  - Détecter les outliers à l’aide de méthodes statistiques (écart interquartile, z-score) ou visuelles (boxplots).  
  - Décider si ces valeurs doivent être corrigées, supprimées ou conservées selon leur impact sur l’analyse.

- **Uniformiser et formater les données :**  
  - Vérifier la cohérence des formats (dates, unités, types de variables).  
  - Nettoyer les doublons, corriger les erreurs typographiques et harmoniser les catégories (ex. : majuscules/minuscules, noms standardisés).


### Interprétation

La réponse ci-dessus provient du provider configuré dans `.env` (`ACTIVE_PROVIDER`).

Pour tester un autre provider, modifiez votre fichier `.env` et redémarrez le kernel, ou instanciez un client avec une configuration explicite :

```python
from config import ProviderConfig, ProviderType

# Exemple pour vLLM
vllm_config = ProviderConfig(
    provider=ProviderType.VLLM,
    model="Qwen/Qwen2.5-72B-Instruct",
    base_url="https://your-vllm-endpoint.com/v1",
    api_key=None
)
vllm_client = LLMClient(vllm_config)
```

## 5. Interface de Chat avec Historique

Le client supporte également une interface de chat avec historique des messages.

In [9]:
# Conversation multi-tours
messages = [
    {"role": "user", "content": "Je veux analyser un dataset de ventes."},
]

response1 = client.chat(messages)
print("Assistant:", response1)

# Ajout de la réponse et continuation
messages.append({"role": "assistant", "content": response1})
messages.append({"role": "user", "content": "Quelles visualisations me recommandes-tu?"})

response2 = client.chat(messages)
print("\nAssistant:", response2)

Assistant: Bien sûr ! Pour analyser un dataset de ventes, voici quelques étapes générales que nous pouvons suivre :

1. **Compréhension du dataset**  
   - Quelles sont les colonnes ou variables présentes ? (ex : date, produit, quantité vendue, prix, région, etc.)  
   - Quel est le volume de données ?  
   - Y a-t-il des données manquantes ou aberrantes ?

2. **Nettoyage des données**  
   - Gérer les valeurs manquantes  
   - Corriger les erreurs ou incohérences  
   - Convertir les types de données si nécessaire (ex : dates en format datetime)

3. **Analyse exploratoire**  
   - Statistiques descriptives (moyenne, médiane, écart-type)  
   - Visualisations (histogrammes, courbes de tendance, heatmaps)  
   - Identification des pics de vente, tendances saisonnières, produits les plus vendus

4. **Analyse approfondie**  
   - Analyse par segment (par produit, région, période)  
   - Analyse des corrélations entre variables  
   - Prévisions ou segmentation clients si besoin

5. **Prés


Assistant: Pour analyser un dataset de ventes, voici quelques visualisations couramment recommandées, selon les objectifs de ton analyse :

### 1. **Visualisations pour les tendances temporelles**  
- **Courbe de tendance (line chart)**  
  Pour visualiser l’évolution des ventes au fil du temps (jour, semaine, mois, trimestre).  
- **Graphique à barres empilées (stacked bar chart)**  
  Pour comparer les ventes de différents produits ou catégories sur une période.  
- **Heatmap temporelle**  
  Pour repérer les jours ou mois avec des ventes particulièrement élevées ou faibles.

### 2. **Analyse produit et catégorie**  
- **Histogramme ou bar chart**  
  Pour voir la répartition des ventes par produit ou catégorie.  
- **Diagramme en secteurs (pie chart)**  
  Pour visualiser la part de marché ou la contribution de chaque produit/catégorie aux ventes totales.

### 3. **Analyse géographique**  
- **Carte choroplèthe**  
  Pour analyser les ventes par région ou zone géographique.

### 4.

## 6. Architecture des Frameworks DS-STAR et MLE-STAR

Ce track AgenticDataScience intègre les frameworks de recherche Google :

### DS-STAR (Data Science Agent)

Architecture Planner-Coder-Verifier pour la data science autonome :

```
┌─────────────┐    ┌─────────────┐    ┌─────────────┐
│  File       │───>│  Planner    │───>│  Coder      │
│  Analyzer   │    │             │    │             │
└─────────────┘    └─────────────┘    └─────────────┘
                          │                   │
                          v                   v
                   ┌─────────────┐    ┌─────────────┐
                   │  Verifier   │<───│  Executor   │
                   │             │    │             │
                   └─────────────┘    └─────────────┘
```

**Performance** : 45.2% accuracy sur DABStep benchmark

### MLE-STAR (ML Engineering Agent)

Extension avec recherche web et optimisation automatique :

- Web Search pour modèles SOTA
- Ablation studies ciblées
- Ensemble strategies automatisées

**Performance** : 63.6% médailles sur MLE-Bench-Lite

## Exercice : Comparaison Multi-Provider

Maintenant que vous avez compris l'architecture, testez votre capacité à utiliser différents providers pour la même tâche.

In [10]:
# Exercice : Comparaison Multi-Provider
# Objectif : Créer une fonction qui compare les réponses de 3 providers sur le même prompt

# TODO: Définissez un prompt de test pertinent pour la data science
test_prompt = None  # Exemple: "Quelles sont les 3 étapes clés pour préparer un dataset pour le ML?"
test_system = None  # Exemple: "Réponds en français de façon concise."

# TODO: Créez une fonction compare_providers qui teste le prompt sur plusieurs providers
def compare_providers(prompt, system_prompt, providers=None):
    """
    Compare les réponses de différents providers pour un prompt donné.
    
    Args:
        prompt: Le prompt utilisateur
        system_prompt: Le prompt système
        providers: Liste de providers à tester (ex: ['gemini', 'vllm'])
    
    Returns:
        dict: Réponses par provider avec temps de réponse
    """
    # Indice: Utilisez ProviderConfig pour configurer chaque provider
    # Indice: Mesurez le temps avec time.time()
    results = {}
    
    # TODO: Implémentez la boucle de test pour chaque provider
    
    return results

# TODO: Exécutez la comparaison et affichez les résultats
# results = compare_providers(test_prompt, test_system)
# for provider, data in results.items():
#     print(f"\n=== {provider.upper()} ({data['time']:.2f}s) ===")
#     print(data['response'])

print("Exercice à compléter - voir les TODO ci-dessus")

Exercice à compléter - voir les TODO ci-dessus


## Résumé et Prochaines Étapes

### Ce que nous avons appris

1. **Configuration multi-provider** : Un seul fichier `.env` permet de switcher entre Gemini, vLLM, OpenAI
2. **Abstraction LiteLLM** : Interface unifiée pour tous les providers
3. **Client LLM simple** : `generate()` et `chat()` pour interagir avec n'importe quel modèle
4. **Architecture DS-STAR** : Framework Planner-Coder-Verifier pour la data science autonome

### Points clés à retenir

| Concept | Description |
|---------|-------------|
| `ProviderConfig` | Configuration d'un provider LLM |
| `LLMClient` | Client unifié pour tous les providers |
| `generate()` | Génération simple avec prompt |
| `chat()` | Conversation multi-tours avec historique |

### Prochaines étapes

- **Lab 9** : Créer un premier agent ADK avec tools Python pour analyser des DataFrames
- **Lab 10** : Implémenter le File Analyzer de DS-STAR
- **Lab 11** : Boucle Planner-Coder-Verifier

---

**Navigation** : [Index](../../README.md) | [Précédent <<](../../PythonAgentsForDataScience/Day3/Labs/Lab7-Data-Analysis-Agent/Lab7-Data-Analysis-Agent.ipynb) | [Suivant >> Lab 9 - First ADK Agent](Lab9-First-ADK-Agent.ipynb)

## Ressources

- [Google ADK Documentation](https://github.com/google/adk-samples)
- [DS-STAR Paper](https://research.google/blog/ds-star-a-state-of-the-art-versatile-data-science-agent/)
- [MLE-STAR Paper](https://research.google/blog/mle-star-a-state-of-the-art-machine-learning-engineering-agents/)
- [LiteLLM Documentation](https://docs.litellm.ai/)